# DICIONÁRIO DE DADOS das bases history.csv e inference.csv (Terra Signal)

Este notebook faz a análise exploratória das duas bases e gera uma tabela de dicionário de dados para cada uma.

## Onde estão os dados
- Catálogo: `hackaton_databricks`
- CSVs: esquema `bases`, volume `bases`
- Tabelas geradas: esquema `tabelas_hackaton`

## Ordem das etapas
1. `raw_history` e `raw_inference`: cópia dos CSVs, tudo como texto
2. Pré-dicionários: perfil dos dados raw
3. `history_tipada` e `inference_tipada`: colunas numéricas convertidas
4. `dicionario_dados_history` e `dicionario_dados_inference`
5. Validações dentro e entre as bases

## Tratamentos
1. **Raw, tudo como texto:** as tabelas raw são criadas com todas as colunas como STRING, sem deixar o Databricks inferir o tipo. Alguns clientes têm CustomerFeedback com mais de um parágrafo no mesmo registro, e a inferência de tipo na importação distorce a tabela. Os tipos são convertidos só na etapa 3, a partir da raw.
2. **Únicas alterações na raw:**
   - Aspas no início e no fim do CustomerFeedback removidas (vinham em alguns textos e em outros não).
   - PhoneService padronizado para Yes/No (vinha também como yes/no).
3. **Tipadas:** tenure e MonthlyIncome como INT; MonthlyCharges e TotalCharges como DECIMAL(10,2), sempre com 2 casas. Os textos unknow (tenure) e null (TotalCharges) viram NULL.

## Informações
- history: 5.634 linhas e 23 colunas. inference: 1.409 linhas e 22 colunas. Churn só existe em history.
- Valores monetários em R$.
- `tipo_variavel` vem do tipo da coluna na tipada: INT = numérica discreta, DECIMAL = numérica contínua, texto = categórica nominal.
- Descrições, dúvidas e confirmações do cliente de cada coluna estão na coluna `observacoes` dos dicionários.
- TotalCharges ÷ MonthlyCharges não dá exatamente o tenure, dá um valor aproximado. A diferença pode vir de multas eventuais ou trocas de plano (inclusão ou exclusão de serviços), mas não é significativa. Exemplo ilustrativo: tenure 50 com cálculo de 51 meses.


**VALIDAÇÕES**

**1. customerID entre history e inference: 0 em comum no ID completo**
- Questionado: o mesmo customerID aparece em history e inference? E a parte numérica (4 primeiros dígitos) ou a de letras (5 últimos), isoladas?
- Resultado: history tem 5.634 IDs distintos e inference 1.409. Em comum: 0 no ID completo e 0 na parte de letras. Na parte numérica, 536 valores aparecem nas duas bases (history tem 4.314 distintos e inference 1.306). Na parte de letras, history tem 5.631 distintos para 5.634 IDs.
- Decisão: sem cliente repetido entre as bases. A parte numérica repete até dentro do history (4.314 distintos para 5.634 IDs), então não identifica o cliente. O customerID completo é a chave.

**2. MultipleLines x PhoneService: 0 inconsistências**
- Questionado: No phone service só aparece quando PhoneService é No, e todo PhoneService No tem MultipleLines No phone service?
- Resultado: em history, PhoneService No tem 563 clientes, todos com No phone service. PhoneService Yes tem 5.071 clientes, com MultipleLines Yes (2.367) ou No (2.704). Em inference, PhoneService No tem 119 clientes, todos com No phone service. PhoneService Yes tem 1.290, com MultipleLines Yes (604) ou No (686). Nenhuma outra combinação.
- Decisão: regra confirmada nas duas bases, sem correção.

**3. Serviços de internet x InternetService: 0 inconsistências**
- Questionado: No internet service só aparece quando InternetService é No, e todo InternetService No tem os seis serviços como No internet service?
- Resultado: em history, InternetService No tem 1.233 clientes, todos com os 6 serviços como No internet service. DSL (1.924) e Fiber optic (2.477) têm 0. Em inference, No tem 293 clientes, todos com 6. DSL (497) e Fiber optic (619) têm 0. Nenhuma outra combinação.
- Decisão: regra confirmada nas duas bases, sem correção.

**4. tenure nulo e tenure 0: 23 nulos com tempo no texto e 11 zeros, todos clientes novos**
- Questionado: os clientes com tenure NULL (unknow) ou 0 são coerentes com MonthlyCharges, TotalCharges e o texto do CustomerFeedback?
- Resultado:
  - tenure 0: 11 clientes (10 em history e 1 em inference). Todos têm "signed up" no texto, e TotalCharges nulo nos 11. Os 10 de history têm Churn No.
  - tenure nulo: 23 clientes (19 em history e 4 em inference). O texto cita o tempo de casa nos 23. Para 22 deles, TotalCharges ÷ MonthlyCharges fica próximo do tempo do texto (ex.: 0135-NMXAP 11,7 contra 12 meses; 3348-CFRNX 28,2 contra 28 meses). O 9651-GTSAQ tem TotalCharges nulo e só o texto informa o tempo ("mais de quatro anos"). Só o 1622-HSHSF tem "signed up" no texto, com tenure estimado 1,0 e texto de 1 mês.
- Decisão: tenure 0 é válido (cliente novo) e fica como 0. Os 23 tenure nulos são valores não gerados na origem, não clientes novos. O tempo pode ser recuperado do texto, usando TotalCharges ÷ MonthlyCharges como conferência. Isso fica para o tratamento, e a tipada mantém NULL. A exceção é o cliente (9651-GTSAQ) que também tem TotalCharges nulo, e texto de feedback falando que já é cliente a mais de 4 anos, ou seja, considerar seu tenure depende exclusivamente do CustomerFeedback.

In [0]:
%sql
-- ============================================================
-- Criação da tabela raw_history (history)
-- ============================================================
-- O que faz: carrega o history.csv do volume e salva como tabela raw_history, com limpeza mínima do texto de feedback e padronização do PhoneService. Em seguida, mostra 10 linhas para conferência.
-- Origem:    /Volumes/hackaton_databricks/bases/bases/history.csv
-- Saída:     tabela hackaton_databricks.tabelas_hackaton.raw_history (23 colunas, uma linha por cliente), mais uma consulta de conferência com até 10 linhas.
-- Regras:    todas as 23 colunas são lidas como STRING. As opções multiLine e escape '"' protegem os textos do CustomerFeedback com quebra de linha ou aspas internas. No CustomerFeedback, as aspas do início e do fim são removidas e as do meio ficam como estão. O texto "null", se existir, não é convertido. PhoneService é padronizado para Yes/No com initcap, pois a coluna tinha No, Yes, no e yes. MonthlyCharges e TotalCharges não são ajustados para 2 casas decimais aqui.

CREATE OR REPLACE TABLE hackaton_databricks.tabelas_hackaton.raw_history AS
SELECT
  customerID, gender, SeniorCitizen, Partner, Dependents, tenure, initcap(PhoneService) AS PhoneService, MultipleLines,
  InternetService, OnlineSecurity, OnlineBackup, DeviceProtection, TechSupport, StreamingTV,
  StreamingMovies, Contract, PaperlessBilling, PaymentMethod, MonthlyCharges, TotalCharges, Churn,
  regexp_replace(CustomerFeedback, '^"+|"+$', '') AS CustomerFeedback,
  MonthlyIncome
FROM read_files(
  '/Volumes/hackaton_databricks/bases/bases/history.csv',
  format => 'csv',
  header => true,
  multiLine => true,
  escape => '"',
  schema => 'customerID string, gender string, SeniorCitizen string, Partner string, Dependents string, tenure string, PhoneService string, MultipleLines string, InternetService string, OnlineSecurity string, OnlineBackup string, DeviceProtection string, TechSupport string, StreamingTV string, StreamingMovies string, Contract string, PaperlessBilling string, PaymentMethod string, MonthlyCharges string, TotalCharges string, Churn string, CustomerFeedback string, MonthlyIncome string'
);

SELECT *
FROM hackaton_databricks.tabelas_hackaton.raw_history
LIMIT 10;

customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,OnlineBackup,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn,CustomerFeedback,MonthlyIncome
4578-PHJYZ,Male,0,Yes,Yes,52,Yes,No,DSL,No,Yes,Yes,Yes,Yes,No,One year,Yes,Electronic check,68.75,3482.85,No,"I have been a customer with this internet provider for over four years now and I have been satisfied with the service. The DSL internet is reliable and I have had no issues with connectivity. The monthly charges of $68.75 are reasonable for the service I receive. I appreciate the convenience of paying with electronic check. Overall, I am happy with my experience and have no plans to switch providers.",6532
6289-CPNLD,Male,0,Yes,Yes,33,Yes,No,DSL,No,No,Yes,Yes,Yes,Yes,One year,Yes,Mailed check,73.9,2405.05,Yes,"I've been a customer with this company for over 2 and a half years now, and unfortunately I have decided to churn. The DSL internet service was reliable for the most part, but the monthly charges of $73.9 were a bit steep for the quality of service I was receiving. Additionally, having to mail in a check for payment was inconvenient and outdated. I ultimately decided to switch to a different provider that offered better value and more modern payment options.",7634
2682-KEVRP,Female,1,No,No,22,Yes,No,No,No internet service,No internet service,No internet service,No internet service,No internet service,No internet service,One year,Yes,Mailed check,20.05,417.0,No,"I have been a customer with this company for 22 months now and I have been satisfied with the service. I have not experienced any issues with billing or service interruptions. The monthly charges are affordable at $20.05 and I appreciate being able to pay by mailing a check. The lack of internet service is not a problem for me as I do not require it. Overall, I am happy with my experience as a customer and have no complaints.",3628
5697-GOMBF,Female,1,Yes,Yes,28,No,No phone service,DSL,No,No,No,No,Yes,No,Month-to-month,Yes,Electronic check,35.9,973.65,No,"I have been a customer with this internet provider for 28 months now and I have been very satisfied with the service. The monthly charges are reasonable at $35.9 and I appreciate the convenience of paying with electronic check. The DSL internet service has been reliable and I have not experienced any issues with connectivity. Overall, I am happy with my experience and have no plans to switch providers.",7851
9717-QEBGU,Male,0,No,No,2,Yes,No,Fiber optic,No,No,No,No,No,No,Month-to-month,Yes,Electronic check,70.3,144.0,No,"I have been using the fiber optic internet service for the past 2 months and I am very satisfied with the speed and reliability. The monthly charges of $70.3 are reasonable for the quality of service I am receiving. The electronic check payment method is convenient and hassle-free. Overall, I have had a positive experience with the service and have no plans to switch providers.",1691
6797-LNAQX,Male,0,Yes,Yes,70,Yes,Yes,Fiber optic,No,Yes,No,No,Yes,Yes,Two year,No,Bank transfer (automatic),98.3,6859.5,Yes,"I have been a customer with this company for 70 months and initially signed up for a two-year contract. I have been using their fiber optic internet service and have been satisfied with the speed and reliability. However, I recently decided to churn because I found a better deal with another provider. The monthly charges of $98.3 were reasonable for the service I received. I appreciated the convenience of automatic bank transfers for payment. Overall, my experience with this company was positive, but I ultimately decided to switch due to cost savings.",5487
3050-GBUSH,Female,0,No,No,3,No,No phone service,DSL,No,No,No,No,Yes,No,Month-to-month,Yes,Credit card (automatic),34.8,113.95,No,"I have been a customer with this DSL internet service provider for 3 months now on a month-to-month contract. I am plea

In [0]:
%sql
-- ============================================================
-- Pré-dicionário de dados (history)
-- ============================================================
-- O que faz: cria o pré-dicionário de dados, com uma linha por coluna de raw_history, para a análise exploratória inicial. Em seguida, mostra o resultado completo para conferência.
-- Origem:    hackaton_databricks.tabelas_hackaton.raw_history
-- Saída:     tabela hackaton_databricks.tabelas_hackaton.predicionario_dados_history (ordem, nome_coluna, total_linhas, qtd_distintos, qtd_nulos, qtd_vazios, qtd_texto_null, valores_distintos, qtd_numericos, minimo, maximo, parece_pk), 23 linhas, mais uma consulta de conferência.
-- Regras:    lê de raw_history (tudo texto) e não altera nenhum valor. qtd_distintos não conta NULL, mas conta '' e 'null' como valores. qtd_nulos é o NULL de verdade, qtd_vazios é '' ou só espaços e qtd_texto_null é o texto 'null' escrito na célula. valores_distintos lista os valores só se qtd_distintos <= 5. qtd_numericos conta os valores que convertem para número, e minimo e maximo são calculados sobre eles (NULL em coluna de texto). parece_pk é 'Sim' quando qtd_distintos = total_linhas, sem nulos e sem vazios. O tipo da variável (nominal, ordinal, discreta, contínua, data) não entra aqui e é definido nos dicionários finais, pelo tipo da coluna.

CREATE OR REPLACE TABLE hackaton_databricks.tabelas_hackaton.predicionario_dados_history AS
WITH longo AS (
  SELECT stack(23,
    1,  'customerID', customerID,
    2,  'gender', gender,
    3,  'SeniorCitizen', SeniorCitizen,
    4,  'Partner', Partner,
    5,  'Dependents', Dependents,
    6,  'tenure', tenure,
    7,  'PhoneService', PhoneService,
    8,  'MultipleLines', MultipleLines,
    9,  'InternetService', InternetService,
    10, 'OnlineSecurity', OnlineSecurity,
    11, 'OnlineBackup', OnlineBackup,
    12, 'DeviceProtection', DeviceProtection,
    13, 'TechSupport', TechSupport,
    14, 'StreamingTV', StreamingTV,
    15, 'StreamingMovies', StreamingMovies,
    16, 'Contract', Contract,
    17, 'PaperlessBilling', PaperlessBilling,
    18, 'PaymentMethod', PaymentMethod,
    19, 'MonthlyCharges', MonthlyCharges,
    20, 'TotalCharges', TotalCharges,
    21, 'Churn', Churn,
    22, 'CustomerFeedback', CustomerFeedback,
    23, 'MonthlyIncome', MonthlyIncome
  ) AS (ordem, coluna, valor)
  FROM hackaton_databricks.tabelas_hackaton.raw_history
)
SELECT
  ordem,
  coluna AS nome_coluna,
  COUNT(*) AS total_linhas,
  COUNT(DISTINCT valor) AS qtd_distintos,
  SUM(CASE WHEN valor IS NULL THEN 1 ELSE 0 END) AS qtd_nulos,
  SUM(CASE WHEN trim(valor) = '' THEN 1 ELSE 0 END) AS qtd_vazios,
  SUM(CASE WHEN trim(valor) = 'null' THEN 1 ELSE 0 END) AS qtd_texto_null,
  CASE WHEN COUNT(DISTINCT valor) <= 5 THEN concat_ws(' | ', sort_array(collect_set(valor))) END AS valores_distintos,
  SUM(CASE WHEN try_cast(valor AS DOUBLE) IS NOT NULL THEN 1 ELSE 0 END) AS qtd_numericos,
  MIN(try_cast(valor AS DOUBLE)) AS minimo,
  MAX(try_cast(valor AS DOUBLE)) AS maximo,
  CASE WHEN COUNT(DISTINCT valor) = COUNT(*)
        AND SUM(CASE WHEN valor IS NULL THEN 1 ELSE 0 END) = 0
        AND SUM(CASE WHEN trim(valor) = '' THEN 1 ELSE 0 END) = 0
       THEN 'Sim' ELSE 'Não' END AS parece_pk
FROM longo
GROUP BY ordem, coluna;

SELECT * FROM hackaton_databricks.tabelas_hackaton.predicionario_dados_history ORDER BY ordem;

ordem,nome_coluna,total_linhas,qtd_distintos,qtd_nulos,qtd_vazios,qtd_texto_null,valores_distintos,qtd_numericos,minimo,maximo,parece_pk
1,customerID,5634,5634,0,0,0,null,0,null,null,Sim
2,gender,5634,2,0,0,0,Female | Male,0,null,null,Não
3,SeniorCitizen,5634,2,0,0,0,0 | 1,5634,0.0,1.0,Não
4,Partner,5634,2,0,0,0,No | Yes,0,null,null,Não
5,Dependents,5634,2,0,0,0,No | Yes,0,null,null,Não
6,tenure,5634,74,0,0,0,null,5615,0.0,72.0,Não
7,PhoneService,5634,2,0,0,0,No | Yes,0,null,null,Não
8,MultipleLines,5634,3,0,0,0,No | No phone service | Yes,0,null,null,Não
9,InternetService,5634,3,0,0,0,DSL | Fiber optic | No,0,null,null,Não
10,OnlineSecurity,5634,3,0,0,0,No | No internet service | Yes,0,null,null,Não


In [0]:
%sql
-- ============================================================
-- Criação da tabela history_tipada (history)
-- ============================================================
-- O que faz: cria a tabela history_tipada a partir de raw_history, convertendo as colunas numéricas para seus tipos. Em seguida, mostra 10 linhas para conferência, com as que têm NULL aparecendo primeiro.
-- Origem:    hackaton_databricks.tabelas_hackaton.raw_history
-- Saída:     tabela hackaton_databricks.tabelas_hackaton.history_tipada (23 colunas, uma linha por cliente), mais uma consulta de conferência com 10 linhas.
-- Regras:    raw_history continua intacta. tenure vira INT e o texto 'unknow' vira NULL. MonthlyCharges e TotalCharges viram DECIMAL(10,2), sempre com 2 casas (73.9 vira 73.90), e o texto 'null' do TotalCharges vira NULL. MonthlyIncome vira INT só se o valor for formado apenas por dígitos; qualquer outro formato vira NULL (esperado: nenhum). PhoneService já vem padronizado para Yes/No de raw_history. Todas as outras colunas continuam STRING, inclusive SeniorCitizen, que é categórica.

CREATE OR REPLACE TABLE hackaton_databricks.tabelas_hackaton.history_tipada AS
SELECT
  customerID, gender, SeniorCitizen, Partner, Dependents,
  CASE WHEN tenure RLIKE '^[0-9]+$' THEN CAST(tenure AS INT) END AS tenure,
  PhoneService, MultipleLines, InternetService, OnlineSecurity, OnlineBackup, DeviceProtection,
  TechSupport, StreamingTV, StreamingMovies, Contract, PaperlessBilling, PaymentMethod,
  try_cast(MonthlyCharges AS DECIMAL(10,2)) AS MonthlyCharges,
  try_cast(TotalCharges AS DECIMAL(10,2)) AS TotalCharges,
  Churn, CustomerFeedback,
  CASE WHEN MonthlyIncome RLIKE '^[0-9]+$' THEN CAST(MonthlyIncome AS INT) END AS MonthlyIncome
FROM hackaton_databricks.tabelas_hackaton.raw_history;

SELECT *
FROM hackaton_databricks.tabelas_hackaton.history_tipada
ORDER BY (tenure IS NULL OR TotalCharges IS NULL) DESC
LIMIT 10;

customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,OnlineBackup,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn,CustomerFeedback,MonthlyIncome
4946-EDSEW,Female,0,Yes,Yes,11,Yes,No,No,No internet service,No internet service,No internet service,No internet service,No internet service,No internet service,Month-to-month,Yes,Mailed check,19.25,null,Yes,"I decided to cancel my service after 11 months because I found a better deal with another provider. The monthly charges were reasonable at $19.25, but I was disappointed that there was no internet service included in my plan. I also found the payment method of mailing a check to be inconvenient compared to other options available. Overall, I was satisfied with the service during my time with the company, but ultimately decided to switch providers for better offerings.",8314
4980-URKXC,Male,0,Yes,No,36,Yes,No,No,No internet service,No internet service,No internet service,No internet service,No internet service,No internet service,One year,No,Bank transfer (automatic),20.85,null,No,"I have been a customer with this company for 3 years now and I have been very satisfied with their service. The automatic bank transfer payment method has been convenient for me and I have never had any issues with billing. Even though I do not use their internet service, I find the monthly charges of $20.85 to be reasonable for the other services I receive. Overall, I have no complaints and I plan to continue using their services in the future.",1408
4951-UKAAQ,Female,0,No,No,4,Yes,Yes,Fiber optic,No,Yes,No,No,No,Yes,Month-to-month,Yes,Electronic check,88.95,null,Yes,"I recently decided to cancel my fiber optic internet service after 4 months of being a customer. While the internet speed and reliability were top-notch, I found the monthly charges of $88.95 to be too expensive for my budget. Additionally, I prefer the flexibility of month-to-month contracts, but I found the payment method of electronic check to be inconvenient. Overall, I was satisfied with the service itself, but the pricing and payment options ultimately led me to churn.",1323
3572-UUHRS,Male,0,No,No,45,Yes,No,No,No internet service,No internet service,No internet service,No internet service,No internet service,No internet service,One year,No,Bank transfer (automatic),18.85,null,No,"I have been a customer with this company for over 3 years now and I have had a positive experience overall. The automatic bank transfer payment method is convenient and hassle-free. Although I do not have internet service with this provider, the monthly charges are affordable at $18.85. I have not experienced any issues with billing or customer service during my 45-month tenure. Overall, I am satisfied with the service and would recommend it to others.",7917
7253-UVNDW,Female,0,No,No,46,Yes,No,DSL,No,No,Yes,Yes,No,No,Two year,No,Credit card (automatic),54.35,null,Yes,"I have been a customer with this company for 46 months on a two-year contract. I have been using their DSL internet service and have been satisfied with the speed and reliability. However, I recently decided to switch providers due to better offers in my area. The monthly charges of $54.35 were reasonable for the service provided. The automatic credit card payment method made it convenient for me to pay my bills on time. Overall, I had a positive experience with this company but decided to churn due to better options available.",5277
9360-AHGNL,Female,1,Yes,No,43,Yes,Yes,Fiber optic,Yes,No,Yes,Yes,Yes,Yes,One year,Yes,Mailed check,109.55,null,Yes,"I have been a customer with this company for over three years now, and I have been mostly satisfied with the fiber optic internet service they provide. However, I recently decided to cancel my subscription due to the high monthly charges of $109.55. I found a better deal with another provider and felt it w

In [0]:
%sql
-- ============================================================
-- Criação da tabela raw_inference (inference)
-- ============================================================
-- O que faz: carrega o inference.csv do volume e salva como tabela raw_inference, com limpeza mínima do texto de feedback e padronização do PhoneService. Em seguida, mostra 10 linhas para conferência.
-- Origem:    /Volumes/hackaton_databricks/bases/bases/inference.csv
-- Saída:     tabela hackaton_databricks.tabelas_hackaton.raw_inference (22 colunas, uma linha por cliente), mais uma consulta de conferência com até 10 linhas.
-- Regras:    todas as 22 colunas são lidas como STRING, e esta base não tem a coluna Churn. A opção multiLine protege os 15 clientes com CustomerFeedback de mais de um parágrafo no mesmo registro, e escape '"' trata aspas duplas dentro de texto entre aspas como parte do texto. No CustomerFeedback, as aspas do início e do fim são removidas (ex.: 0320-JDNQG vem com 3 aspas em cada ponta no .csv) e as do meio ficam como estão. O texto "null" do TotalCharges não é convertido. PhoneService é padronizado para Yes/No com initcap, pois a coluna tinha No, Yes, no e yes. MonthlyCharges e TotalCharges não são ajustados para 2 casas decimais aqui.

CREATE OR REPLACE TABLE hackaton_databricks.tabelas_hackaton.raw_inference AS
SELECT
  customerID, gender, SeniorCitizen, Partner, Dependents, tenure, initcap(PhoneService) AS PhoneService, MultipleLines,
  InternetService, OnlineSecurity, OnlineBackup, DeviceProtection, TechSupport, StreamingTV,
  StreamingMovies, Contract, PaperlessBilling, PaymentMethod, MonthlyCharges, TotalCharges,
  regexp_replace(CustomerFeedback, '^"+|"+$', '') AS CustomerFeedback,
  MonthlyIncome
FROM read_files(
  '/Volumes/hackaton_databricks/bases/bases/inference.csv',
  format => 'csv',
  header => true,
  multiLine => true,
  escape => '"',
  schema => 'customerID string, gender string, SeniorCitizen string, Partner string, Dependents string, tenure string, PhoneService string, MultipleLines string, InternetService string, OnlineSecurity string, OnlineBackup string, DeviceProtection string, TechSupport string, StreamingTV string, StreamingMovies string, Contract string, PaperlessBilling string, PaymentMethod string, MonthlyCharges string, TotalCharges string, CustomerFeedback string, MonthlyIncome string'
);

SELECT *
FROM hackaton_databricks.tabelas_hackaton.raw_inference
LIMIT 10;

customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,OnlineBackup,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,CustomerFeedback,MonthlyIncome
4226-KKDON,Male,0,No,No,8,Yes,No,DSL,No,Yes,No,Yes,Yes,No,Month-to-month,Yes,Electronic check,66.65,520.95,I have been a customer with this DSL provider for 8 months now and I am overall satisfied with the service. The monthly charges are reasonable at $66.65 and I appreciate the convenience of the month-to-month contract. The internet service has been reliable and I have not experienced any issues with connectivity. Using electronic check as my payment method has been smooth and hassle-free. I have not experienced any reason to consider switching providers at this time.,1361
8838-GPHZP,Female,0,No,No,63,Yes,No,No,No internet service,No internet service,No internet service,No internet service,No internet service,No internet service,Two year,Yes,Mailed check,20.6,1298.7,"I have been a customer with this company for over 5 years and have been satisfied with their service. Even though I do not have internet service with them, I find their monthly charges to be very reasonable at $20.6. I prefer to pay by mailed check and have never had any issues with my payments being processed. Overall, I have had a positive experience with this company and have no plans to switch to another provider.",3850
0320-JDNQG,Male,0,Yes,Yes,23,Yes,No,Fiber optic,Yes,No,No,Yes,Yes,Yes,Month-to-month,Yes,Electronic check,99.85,2331.3,"I have been a customer with this fiber optic internet service provider for 23 months on a month-to-month contract. The monthly charges of $99.85 are reasonable for the high-speed internet I receive. However, I recently decided to cancel my service due to moving to a new area where this provider is not available. The process of cancelling was smooth and easy, and I appreciated being able to pay with electronic check. Overall, I was satisfied with the service during my time as a customer.",2736
4837-PZTIC,Female,0,No,No,47,No,No phone service,DSL,Yes,No,Yes,Yes,No,No,Two year,No,Mailed check,41.9,null,"I have been a loyal customer for almost 4 years now and have been satisfied with the DSL internet service provided by this company. The monthly charges are reasonable at $41.9 and I appreciate being able to pay by mailing a check. I have not experienced any issues with my service and have no plans to churn in the near future. Overall, I am happy with the service provided.",1139
7748-UMTRK,Female,1,No,Yes,30,Yes,Yes,Fiber optic,No,No,No,Yes,Yes,Yes,Month-to-month,Yes,Electronic check,101.3,2974.5,"I have been a customer with this company for 30 months on a month-to-month contract with no issues of churn. I am very satisfied with the fiber optic internet service provided, as it is reliable and fast. The monthly charges of $101.30 are reasonable for the quality of service I receive. I appreciate the convenience of paying with an electronic check. Overall, I am happy with my experience and would recommend this company to others.",5211
0823-HSCDJ,Male,1,No,No,52,No,No phone service,DSL,No,Yes,No,No,Yes,Yes,Month-to-month,Yes,Electronic check,49.15,2550.9,"I was a customer with this company for over four years on a month-to-month contract with DSL internet service. The monthly charges were reasonable at $49.15. However, I ultimately decided to churn and cancel my service. The service itself was reliable, but I found better deals with other providers. The payment method of electronic check was convenient and hassle-free. Overall, my experience with this company was satisfactory, but I ultimately chose to switch providers for better pricing.",8422
1085-LDWAM,Female,0,Yes,Yes,69,Yes,No,No,No internet service,No internet service,No internet service,No internet service,No internet service,No internet service,One year,Yes,Credit card (automatic),20.35,1442.65,

In [0]:
%sql
-- ============================================================
-- Pré-dicionário de dados (inference)
-- ============================================================
-- O que faz: cria o pré-dicionário de dados, com uma linha por coluna de raw_inference, para a análise exploratória inicial. Em seguida, mostra o resultado completo para conferência.
-- Origem:    hackaton_databricks.tabelas_hackaton.raw_inference
-- Saída:     tabela hackaton_databricks.tabelas_hackaton.predicionario_dados_inference (ordem, nome_coluna, total_linhas, qtd_distintos, qtd_nulos, qtd_vazios, qtd_texto_null, valores_distintos, qtd_numericos, minimo, maximo, parece_pk), 22 linhas, mais uma consulta de conferência.
-- Regras:    lê de raw_inference (tudo texto) e não altera nenhum valor. qtd_distintos não conta NULL, mas conta '' e 'null' como valores. qtd_nulos é o NULL de verdade, qtd_vazios é '' ou só espaços e qtd_texto_null é o texto 'null' escrito na célula. valores_distintos lista os valores só se qtd_distintos <= 5. qtd_numericos conta os valores que convertem para número, e minimo e maximo são calculados sobre eles (NULL em coluna de texto). parece_pk é 'Sim' quando qtd_distintos = total_linhas, sem nulos e sem vazios. O tipo da variável (nominal, ordinal, discreta, contínua, data) não entra aqui e será definido olhando estes resultados.

CREATE OR REPLACE TABLE hackaton_databricks.tabelas_hackaton.predicionario_dados_inference AS
WITH longo AS (
  SELECT stack(22,
    1,  'customerID', customerID,
    2,  'gender', gender,
    3,  'SeniorCitizen', SeniorCitizen,
    4,  'Partner', Partner,
    5,  'Dependents', Dependents,
    6,  'tenure', tenure,
    7,  'PhoneService', PhoneService,
    8,  'MultipleLines', MultipleLines,
    9,  'InternetService', InternetService,
    10, 'OnlineSecurity', OnlineSecurity,
    11, 'OnlineBackup', OnlineBackup,
    12, 'DeviceProtection', DeviceProtection,
    13, 'TechSupport', TechSupport,
    14, 'StreamingTV', StreamingTV,
    15, 'StreamingMovies', StreamingMovies,
    16, 'Contract', Contract,
    17, 'PaperlessBilling', PaperlessBilling,
    18, 'PaymentMethod', PaymentMethod,
    19, 'MonthlyCharges', MonthlyCharges,
    20, 'TotalCharges', TotalCharges,
    21, 'CustomerFeedback', CustomerFeedback,
    22, 'MonthlyIncome', MonthlyIncome
  ) AS (ordem, coluna, valor)
  FROM hackaton_databricks.tabelas_hackaton.raw_inference
)
SELECT
  ordem,
  coluna AS nome_coluna,
  COUNT(*) AS total_linhas,
  COUNT(DISTINCT valor) AS qtd_distintos,
  SUM(CASE WHEN valor IS NULL THEN 1 ELSE 0 END) AS qtd_nulos,
  SUM(CASE WHEN trim(valor) = '' THEN 1 ELSE 0 END) AS qtd_vazios,
  SUM(CASE WHEN trim(valor) = 'null' THEN 1 ELSE 0 END) AS qtd_texto_null,
  CASE WHEN COUNT(DISTINCT valor) <= 5 THEN concat_ws(' | ', sort_array(collect_set(valor))) END AS valores_distintos,
  SUM(CASE WHEN try_cast(valor AS DOUBLE) IS NOT NULL THEN 1 ELSE 0 END) AS qtd_numericos,
  MIN(try_cast(valor AS DOUBLE)) AS minimo,
  MAX(try_cast(valor AS DOUBLE)) AS maximo,
  CASE WHEN COUNT(DISTINCT valor) = COUNT(*)
        AND SUM(CASE WHEN valor IS NULL THEN 1 ELSE 0 END) = 0
        AND SUM(CASE WHEN trim(valor) = '' THEN 1 ELSE 0 END) = 0
       THEN 'Sim' ELSE 'Não' END AS parece_pk
FROM longo
GROUP BY ordem, coluna;

SELECT * FROM hackaton_databricks.tabelas_hackaton.predicionario_dados_inference ORDER BY ordem;

ordem,nome_coluna,total_linhas,qtd_distintos,qtd_nulos,qtd_vazios,qtd_texto_null,valores_distintos,qtd_numericos,minimo,maximo,parece_pk
1,customerID,1409,1409,0,0,0,null,0,null,null,Sim
2,gender,1409,2,0,0,0,Female | Male,0,null,null,Não
3,SeniorCitizen,1409,2,0,0,0,0 | 1,1409,0.0,1.0,Não
4,Partner,1409,2,0,0,0,No | Yes,0,null,null,Não
5,Dependents,1409,2,0,0,0,No | Yes,0,null,null,Não
6,tenure,1409,74,0,0,0,null,1405,0.0,72.0,Não
7,PhoneService,1409,2,0,0,0,No | Yes,0,null,null,Não
8,MultipleLines,1409,3,0,0,0,No | No phone service | Yes,0,null,null,Não
9,InternetService,1409,3,0,0,0,DSL | Fiber optic | No,0,null,null,Não
10,OnlineSecurity,1409,3,0,0,0,No | No internet service | Yes,0,null,null,Não


In [0]:
%sql
-- ============================================================
-- Criação da tabela inference_tipada (inference)
-- ============================================================
-- O que faz: cria a tabela inference_tipada a partir de raw_inference, convertendo as colunas numéricas para seus tipos. Em seguida, mostra todas as linhas para conferência, com as que têm NULL aparecendo primeiro.
-- Origem:    hackaton_databricks.tabelas_hackaton.raw_inference
-- Saída:     tabela hackaton_databricks.tabelas_hackaton.inference_tipada (22 colunas, uma linha por cliente), mais uma consulta de conferência com todas as linhas.
-- Regras:    raw_inference continua intacta. tenure vira INT e o texto 'unknow' vira NULL. MonthlyCharges e TotalCharges viram DECIMAL(10,2), sempre com 2 casas (73.9 vira 73.90), e o texto 'null' do TotalCharges vira NULL. MonthlyIncome vira INT só se o valor for formado apenas por dígitos; qualquer outro formato vira NULL (esperado: nenhum). PhoneService já vem padronizado para Yes/No de raw_inference. Todas as outras colunas continuam STRING, inclusive SeniorCitizen, que é categórica.

CREATE OR REPLACE TABLE hackaton_databricks.tabelas_hackaton.inference_tipada AS
SELECT
  customerID, gender, SeniorCitizen, Partner, Dependents,
  CASE WHEN tenure RLIKE '^[0-9]+$' THEN CAST(tenure AS INT) END AS tenure,
  PhoneService, MultipleLines, InternetService, OnlineSecurity, OnlineBackup, DeviceProtection,
  TechSupport, StreamingTV, StreamingMovies, Contract, PaperlessBilling, PaymentMethod,
  try_cast(MonthlyCharges AS DECIMAL(10,2)) AS MonthlyCharges,
  try_cast(TotalCharges AS DECIMAL(10,2)) AS TotalCharges,
  CustomerFeedback,
  CASE WHEN MonthlyIncome RLIKE '^[0-9]+$' THEN CAST(MonthlyIncome AS INT) END AS MonthlyIncome
FROM hackaton_databricks.tabelas_hackaton.raw_inference;

SELECT *
FROM hackaton_databricks.tabelas_hackaton.inference_tipada
ORDER BY (tenure IS NULL OR TotalCharges IS NULL) DESC
LIMIT 10;

customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,OnlineBackup,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,CustomerFeedback,MonthlyIncome
8818-XYFCQ,Male,0,Yes,Yes,32,Yes,Yes,Fiber optic,Yes,No,No,No,Yes,Yes,Month-to-month,Yes,Credit card (automatic),99.55,null,"I was a customer with this company for over 2 and a half years on a month-to-month contract. I had fiber optic internet service with a monthly charge of $99.55. I utilized automatic credit card payments for convenience. However, I recently made the decision to churn as I found a better offer with another provider. While the service was reliable and the speeds were good, I ultimately had to make a cost-saving decision. I would recommend this company to others looking for reliable internet service, but I would also suggest they regularly check for better deals to ensure they are getting the best value.",2785
4378-BZYFP,Male,0,No,No,17,Yes,No,No,No internet service,No internet service,No internet service,No internet service,No internet service,No internet service,One year,Yes,Mailed check,20.50,null,"I have been a customer with this company for 17 months now and have been satisfied with the service overall. The monthly charges are affordable at $20.5 and I appreciate the option to pay by mailed check. However, I do not use the internet service offered by the company so I cannot comment on that aspect. Overall, I have had no issues with the service and am happy to continue as a customer.",6644
4001-TSBTV,Female,0,Yes,Yes,58,Yes,No,Fiber optic,No,Yes,Yes,No,Yes,No,Month-to-month,Yes,Electronic check,91.55,null,"I have been a loyal customer of this company for almost 5 years now and I have been very satisfied with their service. The fiber optic internet has been extremely reliable and fast, and I have never experienced any issues with connectivity. The monthly charges are reasonable at $91.55 and I appreciate the convenience of paying through electronic check. Overall, I am happy with my experience and have no plans to switch providers.",5415
2279-AXJJK,Male,0,Yes,No,52,Yes,Yes,Fiber optic,No,Yes,No,Yes,No,Yes,Month-to-month,Yes,Credit card (automatic),95.65,null,"I have been a loyal customer for over 4 years now and I have been extremely satisfied with the fiber optic internet service provided by this company. The automatic credit card payment option makes it convenient for me to pay my monthly bill on time without any hassle. The monthly charges are reasonable for the quality of service I receive. I have never experienced any issues with connectivity or speed, and I appreciate the consistency of the service. Overall, I am very happy with my experience as a customer and would highly recommend this company to others.",1061
6502-HCJTI,Male,1,Yes,No,7,Yes,Yes,Fiber optic,Yes,Yes,No,No,No,Yes,Month-to-month,Yes,Credit card (automatic),94.70,null,"I recently switched internet providers after being with this company for 7 months. The fiber optic service was fast and reliable, but the monthly charges of $94.7 were a bit steep for my budget. The automatic credit card payment method was convenient, but I ultimately decided to switch to a different provider offering a more affordable plan. Overall, I had a positive experience with the internet service itself, but the cost was a deciding factor in my decision to churn.",9947
0259-GBZSH,Male,0,No,No,2,Yes,Yes,Fiber optic,No,No,No,No,Yes,No,Month-to-month,Yes,Electronic check,85.65,null,"I recently signed up for fiber optic internet with this provider on a month-to-month contract. Unfortunately, after just 2 months, I decided to cancel due to some issues with the service. While the speed and reliability were decent, I found the monthly charges of $85.65 to be a bit high compared to other providers in the area. Additionally, I experienced some difficulties with the electronic check payment met

In [0]:
%sql
%sql
-- ============================================================
-- Criação da tabela dicionario_dados_history (history)
-- ============================================================
-- O que faz: cria o dicionário de dados final, com uma linha por coluna de history_tipada, incluindo o tipo da coluna, o tipo da variável e observações por coluna. Em seguida, mostra o resultado completo para conferência.
-- Origem:    hackaton_databricks.tabelas_hackaton.history_tipada (valores) e hackaton_databricks.information_schema.columns (ordem e tipo das colunas)
-- Saída:     tabela hackaton_databricks.tabelas_hackaton.dicionario_dados_history (ordem, nome_coluna, tipo_dado, tipo_variavel, total_linhas, qtd_distintos, qtd_nulos, qtd_vazios, valores_distintos, minimo, maximo, parece_pk, observacoes), 23 linhas, mais uma consulta de conferência.
-- Regras:    tipo_dado é o tipo da coluna na tabela (STRING, INT ou DECIMAL). tipo_variavel segue o tipo_dado: INT é Numérica discreta, DECIMAL é Numérica contínua e STRING é Categórica nominal. qtd_distintos não conta NULL, qtd_nulos é o NULL de verdade e qtd_vazios é '' ou só espaços. valores_distintos lista os valores só se qtd_distintos <= 5. minimo e maximo só aparecem para colunas numéricas. parece_pk é 'Sim' quando qtd_distintos = total_linhas, sem nulos e sem vazios. observacoes vem do texto escrito no bloco VALUES ao final, uma linha por coluna, só onde houver observação. Os NULL de tenure (19 unknow) e de TotalCharges (278 null) vieram da conversão em history_tipada e agora aparecem em qtd_nulos.

CREATE OR REPLACE TABLE hackaton_databricks.tabelas_hackaton.dicionario_dados_history AS
WITH longo AS (
  SELECT stack(23,
    'customerID', CAST(customerID AS STRING),
    'gender', CAST(gender AS STRING),
    'SeniorCitizen', CAST(SeniorCitizen AS STRING),
    'Partner', CAST(Partner AS STRING),
    'Dependents', CAST(Dependents AS STRING),
    'tenure', CAST(tenure AS STRING),
    'PhoneService', CAST(PhoneService AS STRING),
    'MultipleLines', CAST(MultipleLines AS STRING),
    'InternetService', CAST(InternetService AS STRING),
    'OnlineSecurity', CAST(OnlineSecurity AS STRING),
    'OnlineBackup', CAST(OnlineBackup AS STRING),
    'DeviceProtection', CAST(DeviceProtection AS STRING),
    'TechSupport', CAST(TechSupport AS STRING),
    'StreamingTV', CAST(StreamingTV AS STRING),
    'StreamingMovies', CAST(StreamingMovies AS STRING),
    'Contract', CAST(Contract AS STRING),
    'PaperlessBilling', CAST(PaperlessBilling AS STRING),
    'PaymentMethod', CAST(PaymentMethod AS STRING),
    'MonthlyCharges', CAST(MonthlyCharges AS STRING),
    'TotalCharges', CAST(TotalCharges AS STRING),
    'Churn', CAST(Churn AS STRING),
    'CustomerFeedback', CAST(CustomerFeedback AS STRING),
    'MonthlyIncome', CAST(MonthlyIncome AS STRING)
  ) AS (coluna, valor)
   FROM hackaton_databricks.tabelas_hackaton.history_tipada
),
perfil AS (
  SELECT
    coluna,
    COUNT(*) AS total_linhas,
    COUNT(DISTINCT valor) AS qtd_distintos,
    SUM(CASE WHEN valor IS NULL THEN 1 ELSE 0 END) AS qtd_nulos,
    SUM(CASE WHEN trim(valor) = '' THEN 1 ELSE 0 END) AS qtd_vazios,
    CASE WHEN COUNT(DISTINCT valor) <= 5 THEN concat_ws(' | ', sort_array(collect_set(valor))) END AS valores_distintos,
    MIN(try_cast(valor AS DOUBLE)) AS minimo,
    MAX(try_cast(valor AS DOUBLE)) AS maximo,
    CASE WHEN COUNT(DISTINCT valor) = COUNT(*)
          AND SUM(CASE WHEN valor IS NULL THEN 1 ELSE 0 END) = 0
          AND SUM(CASE WHEN trim(valor) = '' THEN 1 ELSE 0 END) = 0
         THEN 'Sim' ELSE 'Não' END AS parece_pk
  FROM longo
  GROUP BY coluna
),
tipos AS (
  SELECT
    column_name,
    ordinal_position,
    data_type,
    CASE WHEN data_type IN ('INT', 'INTEGER', 'BIGINT') THEN 'Numérica discreta'
         WHEN data_type LIKE 'DECIMAL%' THEN 'Numérica contínua'
         ELSE 'Categórica nominal' END AS tipo_variavel
  FROM hackaton_databricks.information_schema.columns
  WHERE table_schema = 'tabelas_hackaton' AND table_name = 'history_tipada'
)
SELECT
  t.ordinal_position AS ordem,
  p.coluna AS nome_coluna,
  t.data_type AS tipo_dado,
  t.tipo_variavel,
  p.total_linhas,
  p.qtd_distintos,
  p.qtd_nulos,
  p.qtd_vazios,
  p.valores_distintos,
  CASE WHEN t.tipo_variavel LIKE 'Numérica%' THEN p.minimo END AS minimo,
  CASE WHEN t.tipo_variavel LIKE 'Numérica%' THEN p.maximo END AS maximo,
  p.parece_pk,
  o.observacoes
FROM perfil p
JOIN tipos t ON t.column_name = p.coluna
LEFT JOIN (VALUES
   ('customerID', 'Descrição: identificador exclusivo por cliente, sem outro significado (confirmado).'),
   ('gender', 'Descrição: gênero do cliente.'),
   ('SeniorCitizen', 'Descrição: identifica se o cliente é idoso (1 = idoso, 0 = não idoso). Cliente: a idade é pouco relevante, não há data de nascimento; pode-se assumir 65 anos como corte.'),
   ('Partner', 'Descrição: cônjuge do cliente, ou seja, se o cliente é casado ou não (confirmado pelo cliente).'),
   ('Dependents', 'Descrição: dependentes do cliente. Dúvida: sem resposta do cliente; checar o que significa, talvez outros moradores na casa, filhos, etc.'),
   ('tenure', 'Descrição: tempo em meses como cliente. Cancelados: da contratação ao cancelamento. Ativos: da contratação até o preenchimento da pesquisa de satisfação (confirmado pelo cliente). 19 valores NULL (escritos unknow), todos inconsistentes: o texto do feedback cita o tempo. Demais entre 0 e 72, sendo zero para 10 cliente de assinatura recente de contrato conforme feedback do mesmo.'),
   ('PhoneService', 'Descrição: identifica se o contrato inclui esse serviço. Valores vinham como Yes/No e yes/no. Padronizado para Yes/No na carga.'),
   ('MultipleLines', 'Descrição: múltiplas linhas telefônicas. Yes = múltiplas; No = linha única; No phone service = sem telefonia (confirmado pelo cliente).'),
   ('InternetService', 'Descrição: identifica se o contrato inclui esse serviço e seu tipo (DSL ou Fiber optic).'),
   ('OnlineSecurity', 'Descrição: serviço de internet contratado (Yes/No). No internet service = sem internet (confirmado pelo cliente).'),
   ('OnlineBackup', 'Descrição: serviço de internet contratado (Yes/No). No internet service = sem internet (confirmado pelo cliente).'),
   ('DeviceProtection', 'Descrição: serviço de internet contratado (Yes/No). No internet service = sem internet (confirmado pelo cliente).'),
   ('TechSupport', 'Descrição: serviço de internet contratado (Yes/No). No internet service = sem internet (confirmado pelo cliente).'),
   ('StreamingTV', 'Descrição: serviço de internet contratado (Yes/No). No internet service = sem internet (confirmado pelo cliente).'),
   ('StreamingMovies', 'Descrição: serviço de internet contratado (Yes/No). No internet service = sem internet (confirmado pelo cliente).'),
   ('Contract', 'Descrição: identifica a duração do contrato (mensal, um ano ou dois anos).'),
   ('PaperlessBilling', 'Descrição: identifica se a cobrança é sem papel ou não.'),
   ('PaymentMethod', 'Descrição: identifica o método de pagamento.'),
   ('MonthlyCharges', 'Descrição: valor total pago no mês, incluindo taxas, encargos e outros serviços. Moeda: R$ (confirmado pelo cliente). Dúvida: os textos de CustomerFeedback citam valores com $; confirmar.'),
   ('TotalCharges', 'Descrição: montante acumulado gasto pelo cliente desde o início do contrato, basicamente a soma do MonthlyCharges. Moeda: R$. Tem 278 valores escritos null (texto), por perda de dados na origem; a estratégia de tratamento cabe a nós.'),
   ('Churn', 'Descrição: Yes = cliente que efetivamente cancelou o contrato, na data em que a base foi construída (foto do momento; pode recontratar ou cancelar depois). Inadimplência não define churn. Coluna exclusiva de history.csv, não tem em inference.csv.'),
   ('CustomerFeedback', 'Descrição: texto digitado ou ditado pelo cliente na pesquisa de satisfação. Sem registro de quando foi preenchido, pode ser de até alguns meses antes das demais colunas; existem todas as combinações entre preencher a pesquisa e cancelar. Dúvida: verificar se há como classificar o risco com as informações dessa coluna.'),
   ('MonthlyIncome', 'Descrição: renda familiar mensal do cliente.')
) AS o(nome_coluna, observacoes)
  ON p.coluna = o.nome_coluna;

SELECT * FROM hackaton_databricks.tabelas_hackaton.dicionario_dados_history ORDER BY ordem;

ordem,nome_coluna,tipo_dado,tipo_variavel,total_linhas,qtd_distintos,qtd_nulos,qtd_vazios,valores_distintos,minimo,maximo,parece_pk,observacoes
0,customerID,STRING,Categórica nominal,5634,5634,0,0,null,null,null,Sim,"Descrição: identificador exclusivo por cliente, sem outro significado (confirmado)."
1,gender,STRING,Categórica nominal,5634,2,0,0,Female | Male,null,null,Não,Descrição: gênero do cliente.
2,SeniorCitizen,STRING,Categórica nominal,5634,2,0,0,0 | 1,null,null,Não,"Descrição: identifica se o cliente é idoso (1 = idoso, 0 = não idoso). Cliente: a idade é pouco relevante, não há data de nascimento; pode-se assumir 65 anos como corte."
3,Partner,STRING,Categórica nominal,5634,2,0,0,No | Yes,null,null,Não,"Descrição: cônjuge do cliente, ou seja, se o cliente é casado ou não (confirmado pelo cliente)."
4,Dependents,STRING,Categórica nominal,5634,2,0,0,No | Yes,null,null,Não,"Descrição: dependentes do cliente. Dúvida: sem resposta do cliente; checar o que significa, talvez outros moradores na casa, filhos, etc."
5,tenure,INT,Numérica discreta,5634,73,19,0,null,0.0,72.0,Não,"Descrição: tempo em meses como cliente. Cancelados: da contratação ao cancelamento. Ativos: da contratação até o preenchimento da pesquisa de satisfação (confirmado pelo cliente). 19 valores NULL (escritos unknow), todos inconsistentes: o texto do feedback cita o tempo. Demais entre 0 e 72, sendo zero para 10 cliente de assinatura recente de contrato conforme feedback do mesmo."
6,PhoneService,STRING,Categórica nominal,5634,2,0,0,No | Yes,null,null,Não,Descrição: identifica se o contrato inclui esse serviço. Valores vinham como Yes/No e yes/no. Padronizado para Yes/No na carga.
7,MultipleLines,STRING,Categórica nominal,5634,3,0,0,No | No phone service | Yes,null,null,Não,Descrição: múltiplas linhas telefônicas. Yes = múltiplas; No = linha única; No phone service = sem telefonia (confirmado pelo cliente).
8,InternetService,STRING,Categórica nominal,5634,3,0,0,DSL | Fiber optic | No,null,null,Não,Descrição: identifica se o contrato inclui esse serviço e seu tipo (DSL ou Fiber optic).
9,OnlineSecurity,STRING,Categórica nominal,5634,3,0,0,No | No internet service | Yes,null,null,Não,Descrição: serviço de internet contratado (Yes/No). No internet service = sem internet (confirmado pelo cliente).


In [0]:
%sql
-- ============================================================
-- Criação da tabela dicionario_dados_inference (inference)
-- ============================================================
-- O que faz: cria o dicionário de dados final, com uma linha por coluna de inference_tipada, incluindo o tipo da coluna, o tipo da variável e observações por coluna. Em seguida, mostra o resultado completo para conferência.
-- Origem:    hackaton_databricks.tabelas_hackaton.inference_tipada (valores) e hackaton_databricks.information_schema.columns (ordem e tipo das colunas)
-- Saída:     tabela hackaton_databricks.tabelas_hackaton.dicionario_dados_inference (ordem, nome_coluna, tipo_dado, tipo_variavel, total_linhas, qtd_distintos, qtd_nulos, qtd_vazios, valores_distintos, minimo, maximo, parece_pk, observacoes), 22 linhas, mais uma consulta de conferência.
-- Regras:    tipo_dado é o tipo da coluna na tabela (STRING, INT ou DECIMAL). tipo_variavel segue o tipo_dado: INT é Numérica discreta, DECIMAL é Numérica contínua e STRING é Categórica nominal. qtd_distintos não conta NULL, qtd_nulos é o NULL de verdade e qtd_vazios é '' ou só espaços. valores_distintos lista os valores só se qtd_distintos <= 5. minimo e maximo só aparecem para colunas numéricas. parece_pk é 'Sim' quando qtd_distintos = total_linhas, sem nulos e sem vazios. observacoes vem do texto escrito no bloco VALUES ao final, uma linha por coluna, só onde houver observação. Os NULL de tenure (4 unknow) e de TotalCharges (64 null) vieram da conversão em inference_tipada e agora aparecem em qtd_nulos. Diferente do pré-dicionário, não há qtd_texto_null nem qtd_numericos, pois o tipo da coluna na tabela já responde isso.

CREATE OR REPLACE TABLE hackaton_databricks.tabelas_hackaton.dicionario_dados_inference AS
WITH longo AS (
  SELECT stack(22,
    'customerID', CAST(customerID AS STRING),
    'gender', CAST(gender AS STRING),
    'SeniorCitizen', CAST(SeniorCitizen AS STRING),
    'Partner', CAST(Partner AS STRING),
    'Dependents', CAST(Dependents AS STRING),
    'tenure', CAST(tenure AS STRING),
    'PhoneService', CAST(PhoneService AS STRING),
    'MultipleLines', CAST(MultipleLines AS STRING),
    'InternetService', CAST(InternetService AS STRING),
    'OnlineSecurity', CAST(OnlineSecurity AS STRING),
    'OnlineBackup', CAST(OnlineBackup AS STRING),
    'DeviceProtection', CAST(DeviceProtection AS STRING),
    'TechSupport', CAST(TechSupport AS STRING),
    'StreamingTV', CAST(StreamingTV AS STRING),
    'StreamingMovies', CAST(StreamingMovies AS STRING),
    'Contract', CAST(Contract AS STRING),
    'PaperlessBilling', CAST(PaperlessBilling AS STRING),
    'PaymentMethod', CAST(PaymentMethod AS STRING),
    'MonthlyCharges', CAST(MonthlyCharges AS STRING),
    'TotalCharges', CAST(TotalCharges AS STRING),
    'CustomerFeedback', CAST(CustomerFeedback AS STRING),
    'MonthlyIncome', CAST(MonthlyIncome AS STRING)
  ) AS (coluna, valor)
  FROM hackaton_databricks.tabelas_hackaton.inference_tipada
),
perfil AS (
  SELECT
    coluna,
    COUNT(*) AS total_linhas,
    COUNT(DISTINCT valor) AS qtd_distintos,
    SUM(CASE WHEN valor IS NULL THEN 1 ELSE 0 END) AS qtd_nulos,
    SUM(CASE WHEN trim(valor) = '' THEN 1 ELSE 0 END) AS qtd_vazios,
    CASE WHEN COUNT(DISTINCT valor) <= 5 THEN concat_ws(' | ', sort_array(collect_set(valor))) END AS valores_distintos,
    MIN(try_cast(valor AS DOUBLE)) AS minimo,
    MAX(try_cast(valor AS DOUBLE)) AS maximo,
    CASE WHEN COUNT(DISTINCT valor) = COUNT(*)
          AND SUM(CASE WHEN valor IS NULL THEN 1 ELSE 0 END) = 0
          AND SUM(CASE WHEN trim(valor) = '' THEN 1 ELSE 0 END) = 0
         THEN 'Sim' ELSE 'Não' END AS parece_pk
  FROM longo
  GROUP BY coluna
),
tipos AS (
  SELECT
    column_name,
    ordinal_position,
    data_type,
    CASE WHEN data_type IN ('INT', 'INTEGER', 'BIGINT') THEN 'Numérica discreta'
         WHEN data_type LIKE 'DECIMAL%' THEN 'Numérica contínua'
         ELSE 'Categórica nominal' END AS tipo_variavel
  FROM hackaton_databricks.information_schema.columns
  WHERE table_schema = 'tabelas_hackaton' AND table_name = 'inference_tipada'
)
SELECT
  t.ordinal_position AS ordem,
  p.coluna AS nome_coluna,
  t.data_type AS tipo_dado,
  t.tipo_variavel,
  p.total_linhas,
  p.qtd_distintos,
  p.qtd_nulos,
  p.qtd_vazios,
  p.valores_distintos,
  CASE WHEN t.tipo_variavel LIKE 'Numérica%' THEN p.minimo END AS minimo,
  CASE WHEN t.tipo_variavel LIKE 'Numérica%' THEN p.maximo END AS maximo,
  p.parece_pk,
  o.observacoes
FROM perfil p
JOIN tipos t ON t.column_name = p.coluna
LEFT JOIN (VALUES
   ('customerID', 'Descrição: identificador exclusivo por cliente, sem outro significado (confirmado).'),
   ('gender', 'Descrição: gênero do cliente.'),
   ('SeniorCitizen', 'Descrição: identifica se o cliente é idoso (1 = idoso, 0 = não idoso). Cliente: a idade é pouco relevante, não há data de nascimento; pode-se assumir 65 anos como corte.'),
   ('Partner', 'Descrição: cônjuge do cliente, ou seja, se o cliente é casado ou não (confirmado pelo cliente).'),
   ('Dependents', 'Descrição: dependentes do cliente. Dúvida: sem resposta do cliente; checar o que significa, talvez outros moradores na casa, filhos, etc.'),
   ('tenure', 'Descrição: tempo em meses como cliente. Cancelados: da contratação ao cancelamento. Ativos: da contratação até o preenchimento da pesquisa de satisfação (confirmado pelo cliente). 4 valores NULL (escritos unknow), todos inconsistentes: o texto do feedback cita o tempo. Demais entre 0 e 72, sendo zero para 01 cliente de assinatura recente de contrato conforme feedback do mesmo'),
   ('PhoneService', 'Descrição: identifica se o contrato inclui esse serviço. Valores vinham como Yes/No e yes/no. Padronizado para Yes/No na carga.'),
   ('MultipleLines', 'Descrição: múltiplas linhas telefônicas. Yes = múltiplas; No = linha única; No phone service = sem telefonia (confirmado pelo cliente).'),
   ('InternetService', 'Descrição: identifica se o contrato inclui esse serviço e seu tipo (DSL ou Fiber optic).'),
   ('OnlineSecurity', 'Descrição: serviço de internet contratado (Yes/No). No internet service = sem internet (confirmado pelo cliente).'),
   ('OnlineBackup', 'Descrição: serviço de internet contratado (Yes/No). No internet service = sem internet (confirmado pelo cliente).'),
   ('DeviceProtection', 'Descrição: serviço de internet contratado (Yes/No). No internet service = sem internet (confirmado pelo cliente).'),
   ('TechSupport', 'Descrição: serviço de internet contratado (Yes/No). No internet service = sem internet (confirmado pelo cliente).'),
   ('StreamingTV', 'Descrição: serviço de internet contratado (Yes/No). No internet service = sem internet (confirmado pelo cliente).'),
   ('StreamingMovies', 'Descrição: serviço de internet contratado (Yes/No). No internet service = sem internet (confirmado pelo cliente).'),
   ('Contract', 'Descrição: identifica a duração do contrato (mensal, um ano ou dois anos).'),
   ('PaperlessBilling', 'Descrição: identifica se a cobrança é sem papel ou não.'),
   ('PaymentMethod', 'Descrição: identifica o método de pagamento.'),
   ('MonthlyCharges', 'Descrição: valor total pago no mês, incluindo taxas, encargos e outros serviços. Moeda: R$ (confirmado pelo cliente). Dúvida: os textos de CustomerFeedback citam valores com $; confirmar.'),
   ('TotalCharges', 'Descrição: montante acumulado gasto pelo cliente desde o início do contrato, basicamente a soma do MonthlyCharges. Moeda: R$. Tem 278 valores escritos null (texto), por perda de dados na origem; a estratégia de tratamento cabe a nós.'),
   ('Churn', 'Descrição: Yes = cliente que efetivamente cancelou o contrato, na data em que a base foi construída (foto do momento; pode recontratar ou cancelar depois). Inadimplência não define churn. Coluna exclusiva de history.csv, não tem em inference.csv.'),
   ('CustomerFeedback', 'Descrição: texto digitado ou ditado pelo cliente na pesquisa de satisfação. Sem registro de quando foi preenchido, pode ser de até alguns meses antes das demais colunas; existem todas as combinações entre preencher a pesquisa e cancelar. Dúvida: verificar se há como classificar o risco com as informações dessa coluna.'),
   ('MonthlyIncome', 'Descrição: renda familiar mensal do cliente.')
) AS o(nome_coluna, observacoes)
  ON p.coluna = o.nome_coluna;

SELECT * FROM hackaton_databricks.tabelas_hackaton.dicionario_dados_inference ORDER BY ordem;

ordem,nome_coluna,tipo_dado,tipo_variavel,total_linhas,qtd_distintos,qtd_nulos,qtd_vazios,valores_distintos,minimo,maximo,parece_pk,observacoes
0,customerID,STRING,Categórica nominal,1409,1409,0,0,null,null,null,Sim,"Descrição: identificador exclusivo por cliente, sem outro significado (confirmado)."
1,gender,STRING,Categórica nominal,1409,2,0,0,Female | Male,null,null,Não,Descrição: gênero do cliente.
2,SeniorCitizen,STRING,Categórica nominal,1409,2,0,0,0 | 1,null,null,Não,"Descrição: identifica se o cliente é idoso (1 = idoso, 0 = não idoso). Cliente: a idade é pouco relevante, não há data de nascimento; pode-se assumir 65 anos como corte."
3,Partner,STRING,Categórica nominal,1409,2,0,0,No | Yes,null,null,Não,"Descrição: cônjuge do cliente, ou seja, se o cliente é casado ou não (confirmado pelo cliente)."
4,Dependents,STRING,Categórica nominal,1409,2,0,0,No | Yes,null,null,Não,"Descrição: dependentes do cliente. Dúvida: sem resposta do cliente; checar o que significa, talvez outros moradores na casa, filhos, etc."
5,tenure,INT,Numérica discreta,1409,73,4,0,null,0.0,72.0,Não,"Descrição: tempo em meses como cliente. Cancelados: da contratação ao cancelamento. Ativos: da contratação até o preenchimento da pesquisa de satisfação (confirmado pelo cliente). 4 valores NULL (escritos unknow), todos inconsistentes: o texto do feedback cita o tempo. Demais entre 0 e 72, sendo zero para 01 cliente de assinatura recente de contrato conforme feedback do mesmo"
6,PhoneService,STRING,Categórica nominal,1409,2,0,0,No | Yes,null,null,Não,Descrição: identifica se o contrato inclui esse serviço. Valores vinham como Yes/No e yes/no. Padronizado para Yes/No na carga.
7,MultipleLines,STRING,Categórica nominal,1409,3,0,0,No | No phone service | Yes,null,null,Não,Descrição: múltiplas linhas telefônicas. Yes = múltiplas; No = linha única; No phone service = sem telefonia (confirmado pelo cliente).
8,InternetService,STRING,Categórica nominal,1409,3,0,0,DSL | Fiber optic | No,null,null,Não,Descrição: identifica se o contrato inclui esse serviço e seu tipo (DSL ou Fiber optic).
9,OnlineSecurity,STRING,Categórica nominal,1409,3,0,0,No | No internet service | Yes,null,null,Não,Descrição: serviço de internet contratado (Yes/No). No internet service = sem internet (confirmado pelo cliente).


In [0]:
%sql
-- ============================================================
-- Validação 1: customerID repetido entre history e inference
-- ============================================================
-- O que faz: compara o customerID completo, a parte numérica (4 primeiros dígitos) e a parte de letras (5 últimos), no formato NNNN-LLLLL, entre history e inference, para ver quantos valores aparecem nas duas bases.
-- Origem:    history_tipada e inference_tipada
-- Saída:     consulta de uma linha (ids_history, ids_inference, ids_em_comum, num_history, num_inference, num_em_comum, letras_history, letras_inference, letras_em_comum), sem criar tabela.
-- Regras:    conta valores distintos em cada base e em comum, para as três comparações (código completo, parte numérica e parte de letras). Os campos *_em_comum mostram quantos valores distintos aparecem nas duas bases.
-- ============================================================

WITH h AS (
  SELECT customerID,
         substring(customerID, 1, 4) AS parte_num,
         substring(customerID, 6, 5) AS parte_letras
  FROM hackaton_databricks.tabelas_hackaton.history_tipada
),
i AS (
  SELECT customerID,
         substring(customerID, 1, 4) AS parte_num,
         substring(customerID, 6, 5) AS parte_letras
  FROM hackaton_databricks.tabelas_hackaton.inference_tipada
)
SELECT
  (SELECT COUNT(DISTINCT customerID) FROM h) AS ids_history,
  (SELECT COUNT(DISTINCT customerID) FROM i) AS ids_inference,
  (SELECT COUNT(DISTINCT h.customerID) FROM h JOIN i ON h.customerID = i.customerID) AS ids_em_comum,
  (SELECT COUNT(DISTINCT parte_num) FROM h) AS num_history,
  (SELECT COUNT(DISTINCT parte_num) FROM i) AS num_inference,
  (SELECT COUNT(DISTINCT h.parte_num) FROM h JOIN i ON h.parte_num = i.parte_num) AS num_em_comum,
  (SELECT COUNT(DISTINCT parte_letras) FROM h) AS letras_history,
  (SELECT COUNT(DISTINCT parte_letras) FROM i) AS letras_inference,
  (SELECT COUNT(DISTINCT h.parte_letras) FROM h JOIN i ON h.parte_letras = i.parte_letras) AS letras_em_comum;

ids_history,ids_inference,ids_em_comum,num_history,num_inference,num_em_comum,letras_history,letras_inference,letras_em_comum
5634,1409,0,4314,1306,536,5631,1409,0


In [0]:
%sql
-- ============================================================
-- Validação 2: MultipleLines x PhoneService
-- ============================================================
-- O que faz: conta cada combinação de PhoneService e MultipleLines, por base, para conferir se No phone service só aparece quando PhoneService é No, e se todo PhoneService No tem MultipleLines No phone service.
-- Origem:    history_tipada e inference_tipada
-- Saída:     consulta (base, PhoneService, MultipleLines, qtd), sem criar tabela.
-- Regras:    esperado: PhoneService No só com MultipleLines No phone service, e PhoneService Yes só com MultipleLines Yes ou No. Qualquer outra combinação indica inconsistência.
-- ============================================================.

SELECT 'history' AS base, PhoneService, MultipleLines, COUNT(*) AS qtd
FROM hackaton_databricks.tabelas_hackaton.history_tipada
GROUP BY PhoneService, MultipleLines
UNION ALL
SELECT 'inference' AS base, PhoneService, MultipleLines, COUNT(*) AS qtd
FROM hackaton_databricks.tabelas_hackaton.inference_tipada
GROUP BY PhoneService, MultipleLines
ORDER BY base, PhoneService, MultipleLines;

base,PhoneService,MultipleLines,qtd
history,No,No phone service,563
history,Yes,No,2704
history,Yes,Yes,2367
inference,No,No phone service,119
inference,Yes,No,686
inference,Yes,Yes,604


In [0]:
%sql
-- ============================================================
-- Validação 3: serviços de internet x InternetService
-- ============================================================
-- O que faz: conta, por base e por InternetService, quantos dos seis serviços (OnlineSecurity, OnlineBackup, DeviceProtection, TechSupport, StreamingTV, StreamingMovies) estão como No internet service, para conferir se esse valor só aparece quando InternetService é No, e se todo InternetService No tem os seis serviços como No internet service.
-- Origem:    history_tipada e inference_tipada
-- Saída:     consulta (base, InternetService, servicos_sem_internet, qtd), sem criar tabela.
-- Regras:    servicos_sem_internet é a soma de quantos dos seis serviços estão como No internet service. Esperado: InternetService No só com 6; DSL e Fiber optic só com 0. Qualquer outra combinação quebra a regra.
-- ============================================================

SELECT 'history' AS base, InternetService,
  (CASE WHEN OnlineSecurity = 'No internet service' THEN 1 ELSE 0 END
 + CASE WHEN OnlineBackup = 'No internet service' THEN 1 ELSE 0 END
 + CASE WHEN DeviceProtection = 'No internet service' THEN 1 ELSE 0 END
 + CASE WHEN TechSupport = 'No internet service' THEN 1 ELSE 0 END
 + CASE WHEN StreamingTV = 'No internet service' THEN 1 ELSE 0 END
 + CASE WHEN StreamingMovies = 'No internet service' THEN 1 ELSE 0 END) AS servicos_sem_internet,
  COUNT(*) AS qtd
FROM hackaton_databricks.tabelas_hackaton.history_tipada
GROUP BY 1, 2, 3
UNION ALL
SELECT 'inference' AS base, InternetService,
  (CASE WHEN OnlineSecurity = 'No internet service' THEN 1 ELSE 0 END
 + CASE WHEN OnlineBackup = 'No internet service' THEN 1 ELSE 0 END
 + CASE WHEN DeviceProtection = 'No internet service' THEN 1 ELSE 0 END
 + CASE WHEN TechSupport = 'No internet service' THEN 1 ELSE 0 END
 + CASE WHEN StreamingTV = 'No internet service' THEN 1 ELSE 0 END
 + CASE WHEN StreamingMovies = 'No internet service' THEN 1 ELSE 0 END) AS servicos_sem_internet,
  COUNT(*) AS qtd
FROM hackaton_databricks.tabelas_hackaton.inference_tipada
GROUP BY 1, 2, 3
ORDER BY base, InternetService, servicos_sem_internet;

base,InternetService,servicos_sem_internet,qtd
history,DSL,0,1924
history,Fiber optic,0,2477
history,No,6,1233
inference,DSL,0,497
inference,Fiber optic,0,619
inference,No,6,293


In [0]:
%sql
-- ============================================================
-- Validação 4: tenure nulo (unknow) e tenure 0
-- ============================================================
-- O que faz: lista os clientes com tenure NULL ou 0, nas duas bases, para conferir MonthlyCharges, TotalCharges e o texto do CustomerFeedback.
-- Origem:    history_tipada e inference_tipada
-- Saída:     consulta (base, caso, customerID, tenure, MonthlyCharges, TotalCharges, tenure_estimado, texto_signed_up, Churn, inicio_texto), sem criar tabela.
-- Regras:    caso é 'tenure nulo' ou 'tenure 0'. tenure_estimado é TotalCharges ÷ MonthlyCharges, arredondado a 1 casa (NULL se TotalCharges for nulo), e serve de referência aproximada: o tenure costuma ser próximo, mas nem sempre igual. texto_signed_up é true se o texto tem "signed up" (recently signed up ou just signed up). inicio_texto traz os 150 primeiros caracteres do CustomerFeedback. Churn é NULL na inference, que não tem essa coluna. Esperado: tenure 0 com texto_signed_up verdadeiro. Se texto_signed_up for falso, o tenure deveria ter um valor; sem ele, é inconsistência da base (valor não gerado).
-- ============================================================

SELECT
  'history' AS base,
  CASE WHEN tenure IS NULL THEN 'tenure nulo' ELSE 'tenure 0' END AS caso,
  customerID, tenure, MonthlyCharges, TotalCharges,
  ROUND(TotalCharges / MonthlyCharges, 1) AS tenure_estimado,
  lower(CustomerFeedback) LIKE '%signed up%' AS texto_signed_up,
  Churn,
  LEFT(CustomerFeedback, 150) AS inicio_texto
FROM hackaton_databricks.tabelas_hackaton.history_tipada
WHERE tenure IS NULL OR tenure = 0
UNION ALL
SELECT
  'inference' AS base,
  CASE WHEN tenure IS NULL THEN 'tenure nulo' ELSE 'tenure 0' END AS caso,
  customerID, tenure, MonthlyCharges, TotalCharges,
  ROUND(TotalCharges / MonthlyCharges, 1) AS tenure_estimado,
  lower(CustomerFeedback) LIKE '%signed up%' AS texto_signed_up,
  NULL AS Churn,
  LEFT(CustomerFeedback, 150) AS inicio_texto
FROM hackaton_databricks.tabelas_hackaton.inference_tipada
WHERE tenure IS NULL OR tenure = 0
ORDER BY base, caso, customerID;

base,caso,customerID,tenure,MonthlyCharges,TotalCharges,tenure_estimado,texto_signed_up,Churn,inicio_texto
history,tenure 0,1371-DWPAZ,0,56.05,null,null,true,No,I recently signed up for the two year DSL internet service with a monthly charge of $56.05. The automatic credit card payment method makes it convenie
history,tenure 0,2520-SGTTA,0,20.00,null,null,true,No,"I recently signed up for a two-year contract with this company and so far, I am satisfied with the service. Even though I do not have internet service"
history,tenure 0,2775-SEFEE,0,61.90,null,null,true,No,I recently signed up for a two-year contract with this DSL internet service provider and so far I am very satisfied with the service. The monthly char
history,tenure 0,2923-ARZLG,0,19.70,null,null,true,No,"I recently signed up for a one-year contract with this service provider and so far, I have been satisfied with their service. Despite not having inter"
history,tenure 0,3115-CZMZD,0,20.25,null,null,true,No,"I recently signed up for a two-year contract with this company and so far, I am happy with the service. The monthly charges are affordable at $20.25 a"
history,tenure 0,3213-VVOLG,0,25.35,null,null,true,No,"I recently signed up for a two-year contract with this company and so far, I have been pleased with the service. Even though I do not have internet se"
history,tenure 0,4075-WKNIU,0,73.35,null,null,true,No,"I recently signed up for a two-year contract for DSL internet service with a monthly charge of $73.35. As a new customer with no prior tenure, I have"
history,tenure 0,4367-NUYAO,0,25.75,null,null,true,No,"I recently signed up for a two-year contract with this service provider and so far, I am satisfied with the monthly charges of $25.75. However, I am d"
history,tenure 0,4472-LVYGI,0,52.55,null,null,true,No,I recently signed up for DSL internet service with a two-year contract and opted for automatic bank transfer for payments. The monthly charges are ver
history,tenure 0,7644-OMVMY,0,19.85,null,null,true,No,I recently signed up for a two-year contract with this company and so far I have been satisfied with the service. Even though I don't have internet se
